# 🫁 Lung Segmentation Mask Pre-computation & Tuning (CheXpert)

**Purpose:** Pre-compute lung segmentation masks for the CheXpert-Small dataset and save them to disk for reuse.

**Features:**
- 📁 Downloads CheXpert-Small and filters for Frontal X-rays
- 💾 Saves pixel-level masks (`.npy`) and patch-level masks to disk in `./lung_masks_chexpert`

In [1]:
# ============================================
# 📦 Step 1: Import Libraries
# ============================================

import warnings
warnings.filterwarnings('ignore')

import os, sys, glob, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import cv2

try:
    import kagglehub
except ImportError:
    pass

IN_KAGGLE = os.path.exists('/kaggle/input')

print("✅ All libraries imported")

✅ All libraries imported


In [7]:
# ============================================
# 📁 Step 2: Download and Load CheXpert Dataset
# ============================================

if IN_KAGGLE:
    BASE_PATH = Path('/kaggle/input/chexpert/CheXpert-v1.0-small')
else:
    path = kagglehub.dataset_download("ashery/chexpert")
    BASE_PATH = Path(path)

print(f"📂 Base path: {BASE_PATH}")

# Load metadata
train_df = pd.read_csv(BASE_PATH / 'train.csv')
valid_df = pd.read_csv(BASE_PATH / 'valid.csv')
df = pd.concat([train_df, valid_df], ignore_index=True)

# Filter for Frontal X-rays only
df = df[df['Frontal/Lateral'] == 'Frontal'].copy()

# Fix image paths (remove 'CheXpert-v1.0-small/' from the beginning if present)
def map_path(p):
    p = str(p)
    if p.startswith('CheXpert-v1.0-small/'):
        p = p.replace('CheXpert-v1.0-small/', '', 1)
    return str(BASE_PATH / p)

df['Image Path'] = df['Path'].apply(map_path)
df = df.reset_index(drop=True)

print(f"✅ Loaded {len(df):,} Frontal images from CheXpert")

📂 Base path: /root/.cache/kagglehub/datasets/ashery/chexpert/versions/1
✅ Loaded 191,229 Frontal images from CheXpert


In [8]:
# ============================================
# ⚙️ Step 3: Segmentation Configuration
# ============================================
class SegConfig:
    img_size = 224                  
    use_otsu = True                 
    fixed_threshold = 127           
    morph_kernel_size = 7           
    morph_iterations = 1            
    min_lung_area_ratio = 0.01      
    max_components = 2              
    central_margin = 0.15          
    adaptive_contrast = True           
    contrast_reference_std = 50.0      
    contrast_scale = 0.5               
    use_clahe = True                   
    clahe_clip_limit = 2.0             
    clahe_grid_size = 8                
    gaussian_blur_size = 11         
    mask_binarize_threshold = 0.3   
    patch_size = 16                 
    patch_lung_threshold = 0.3     
    save_pixel_masks = True         
    save_patch_masks = True         
    mask_dtype = np.uint8           

cfg = SegConfig()

if IN_KAGGLE:
    MASK_OUTPUT_DIR = '/kaggle/working/lung_masks_chexpert'
else:
    MASK_OUTPUT_DIR = './lung_masks_chexpert'

PIXEL_MASK_DIR = os.path.join(MASK_OUTPUT_DIR, 'pixel_masks')
PATCH_MASK_DIR = os.path.join(MASK_OUTPUT_DIR, 'patch_masks')

os.makedirs(PIXEL_MASK_DIR, exist_ok=True)
os.makedirs(PATCH_MASK_DIR, exist_ok=True)

print("✅ Segmentation configuration ready")
print(f"   Output dir: {MASK_OUTPUT_DIR}")

✅ Segmentation configuration ready
   Output dir: ./lung_masks_chexpert


In [9]:
# ============================================
# 🫁 Step 4: Configurable Lung Segmentation Function
# ============================================
def segment_lungs(image, config=None):
    if config is None: config = cfg
    if image.dtype != np.uint8:
        img = (image * 255).astype(np.uint8)
    else:
        img = image.copy()

    if config.use_clahe:
        clahe = cv2.createCLAHE(clipLimit=config.clahe_clip_limit, tileGridSize=(config.clahe_grid_size, config.clahe_grid_size))
        img = clahe.apply(img)

    blurred = cv2.GaussianBlur(img, (5, 5), 0)

    if config.use_otsu:
        otsu_val, _ = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if config.adaptive_contrast:
            std_val = np.std(blurred)
            shift = (std_val - config.contrast_reference_std) * config.contrast_scale
            adjusted = int(np.clip(otsu_val - shift, 40, 220))
            _, thresh = cv2.threshold(blurred, adjusted, 255, cv2.THRESH_BINARY_INV)
        else:
            _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        _, thresh = cv2.threshold(blurred, config.fixed_threshold, 255, cv2.THRESH_BINARY_INV)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh, connectivity=8)
    if num_labels <= 1: return np.ones_like(img, dtype=np.float32)

    total_area = img.shape[0] * img.shape[1]
    min_area = total_area * config.min_lung_area_ratio
    areas = stats[1:, cv2.CC_STAT_AREA]
    valid_idx = np.where(areas >= min_area)[0]
    if len(valid_idx) == 0: return np.ones_like(img, dtype=np.float32)

    if config.central_margin > 0:
        H, W = img.shape
        mh, mw = int(H * config.central_margin), int(W * config.central_margin)
        central_zone = np.zeros_like(img, dtype=bool)
        central_zone[mh:H-mh, mw:W-mw] = True
        central_valid = [idx for idx in valid_idx if np.any((labels == (idx + 1)) & central_zone)]
        if len(central_valid) > 0: valid_idx = np.array(central_valid)

    sorted_idx = valid_idx[np.argsort(areas[valid_idx])[::-1]]
    keep = sorted_idx[:config.max_components]
    mask = np.zeros_like(img, dtype=np.uint8)
    for k in keep: mask[labels == (k + 1)] = 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (config.morph_kernel_size, config.morph_kernel_size))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=config.morph_iterations)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=config.morph_iterations)

    if config.gaussian_blur_size > 0:
        mask = cv2.GaussianBlur(mask, (config.gaussian_blur_size, config.gaussian_blur_size), 0)

    mask = (mask / 255.0).astype(np.float32)
    mask = (mask > config.mask_binarize_threshold).astype(np.float32)
    return mask

def lung_mask_to_patch_mask(lung_mask, patch_size=16, img_size=224, threshold=0.3):
    h_patches = img_size // patch_size
    w_patches = img_size // patch_size
    reshaped = lung_mask.reshape(h_patches, patch_size, w_patches, patch_size)
    patch_means = reshaped.transpose(0, 2, 1, 3).reshape(h_patches * w_patches, -1).mean(axis=1)
    return (patch_means > threshold).astype(np.float32)

def load_and_preprocess(img_path, img_size=224):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    return img.astype(np.float32) / 255.0

print("✅ Segmentation functions defined")

✅ Segmentation functions defined


In [10]:
# ============================================
# 💾 Step 5: Batch Pre-Compute & Save All Masks to Disk
# ============================================
print(f"💾 Saving masks to: {MASK_OUTPUT_DIR}")

failed_images = []
total = len(df)

for i in tqdm(range(total), desc="Pre-computing masks", mininterval=2.0):
    # Generate unique mask name based on path to avoid collisions
    # e.g. train/patient00001/study1/view1_frontal.jpg -> patient00001_study1_view1_frontal
    raw_path = df.iloc[i]['Path'].replace('CheXpert-v1.0-small/', '')
    mask_name = raw_path.replace('/', '_').replace('.jpg', '')
    img_path = df.iloc[i]['Image Path']

    try:
        img = load_and_preprocess(img_path, cfg.img_size)
        pixel_mask = segment_lungs(img, config=cfg)

        if cfg.save_pixel_masks:
            out_path = os.path.join(PIXEL_MASK_DIR, f"{mask_name}.npy")
            np.save(out_path, pixel_mask.astype(cfg.mask_dtype))
        
        if cfg.save_patch_masks:
            patch_mask = lung_mask_to_patch_mask(
                pixel_mask, 
                patch_size=cfg.patch_size, 
                img_size=cfg.img_size, 
                threshold=cfg.patch_lung_threshold
            )
            p_out_path = os.path.join(PATCH_MASK_DIR, f"{mask_name}.npy")
            np.save(p_out_path, patch_mask.astype(np.float32))
            
    except Exception as e:
        failed_images.append((mask_name, str(e)))

print(f"\n✅ Finished processing {total} images.")
if failed_images:
    print(f"⚠️ Failed to process {len(failed_images)} images.")

💾 Saving masks to: ./lung_masks_chexpert


Pre-computing masks: 100%|██████████| 191229/191229 [51:42<00:00, 61.64it/s] 


✅ Finished processing 191229 images.
